
# GC Survey / Observatory Analysis

Repository: <a href="https://github.com/Pommers/gc_surveys_telescope">github.com/Pommers/gc_surveys_telescope</a> <br>

# Main Notebook

## Purpose

This notebook performs an analysis on 8 contemporary observatory setups for GC survey capability. It will

1. Create a dataframe of initial observatory parameters (from literature).
2. Generate GC modelling machinery
3. Add GC angular-size calculation
4. Normalize Band-passes for instruments
5. Add filter-specific columns


In [1]:

from pathlib import Path
import sys
import logging
import warnings

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table, vstack
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_area
from astropy.stats import sigma_clipped_stats
from astropy.convolution import Gaussian2DKernel, convolve

from photutils.segmentation import SourceCatalog, detect_sources

from scipy.ndimage import (
    median_filter,
    binary_dilation,
    distance_transform_edt,
)

%load_ext autoreload
%autoreload 2

def find_project_root(marker="src"):
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Could not find project root containing '{marker}'"
    )

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
# CATALOGUE_DIR = DATA_DIR / "catalogues" / "detection"
FIGURE_DIR = PROJECT_ROOT / "figures" 
CONFIG_DIR = PROJECT_ROOT / "config"

# CATALOGUE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from src.io_utils import setup_logging, load_config

setup_logging(verbose=True, name="Coma_GCs_survey")
logger = logging.getLogger("Coma_GCs_survey")

config = load_config(PROJECT_ROOT)

logger.info("Project root: %s", PROJECT_ROOT)
# logger.info("Catalogue directory: %s", CATALOGUE_DIR)
logger.info("Figure directory: %s", FIGURE_DIR)


Project root: /Users/richard/PycharmProjects/gc_surveys_telescope
Figure directory: /Users/richard/PycharmProjects/gc_surveys_telescope/figures


# Create Instrument Dataframe

## Manual data extract from literature

In [ ]:
import pandas as pd
import numpy as np

instrument_data = [
    {
        "facility": "HST",
        "instrument": "ACS/WFC",
        "regime": "targeted_space",
        "observing_mode": "directed",
        "status": "operational",

        "aperture_m": 2.4,
        "fov_width_arcsec": 202.0,
        "fov_height_arcsec": 202.0,
        "fov_area_deg2": (202.0 * 202.0) / 3600**2,
        "pixel_scale_arcsec": 0.049,
        "psf_fwhm_arcsec": 0.09,

        "wavelength_min_um": 0.35,
        "wavelength_max_um": 1.05,
        "gc_filter": "F814W",

        "survey_area_deg2": np.nan,
        "nominal_exposure_s": np.nan,
        "survey_depth_ab": np.nan,
        "survey_depth_snr": np.nan,

        "reference": "Sirianni_2005",
        "notes": "Directed observations; PSF is approximate and filter dependent."
    },

    {
        "facility": "JWST",
        "instrument": "NIRCam SW",
        "regime": "targeted_space",
        "observing_mode": "directed",
        "status": "operational",

        "aperture_m": 6.5,
        "fov_width_arcsec": np.nan,
        "fov_height_arcsec": np.nan,
        "fov_area_deg2": 9.7 / 3600.0,  # two-module imaging area
        "pixel_scale_arcsec": 0.031,
        "psf_fwhm_arcsec": 0.05,

        "wavelength_min_um": 0.60,
        "wavelength_max_um": 2.30,
        "gc_filter": "F150W",

        "survey_area_deg2": np.nan,
        "nominal_exposure_s": np.nan,
        "survey_depth_ab": np.nan,
        "survey_depth_snr": np.nan,

        "reference": "Rieke_2023",
        "notes": "NIRCam short-wavelength channel; PSF approximate at F150W."
    },

    {
        "facility": "Subaru",
        "instrument": "HSC",
        "regime": "wide_ground",
        "observing_mode": "survey+directed",
        "status": "operational",

        "aperture_m": 8.2,
        "fov_width_arcsec": 1.5 * 3600,  # diameter
        "fov_height_arcsec": 1.5 * 3600,
        "fov_area_deg2": np.pi * (1.5 / 2)**2,
        "pixel_scale_arcsec": 0.168,
        "psf_fwhm_arcsec": 0.60,

        "wavelength_min_um": np.nan,
        "wavelength_max_um": np.nan,
        "gc_filter": "i",

        "survey_area_deg2": 1400.0,
        "nominal_exposure_s": 1200.0,  # nominal SSP exposure in i
        "survey_depth_ab": np.nan,      # fill with i-band value, not r-band
        "survey_depth_snr": 5.0,

        "reference": "Miyazaki_2018; Aihara_2018",
        "notes": "HSC-SSP Wide. Median i-band seeing ~0.6 arcsec; nominal i exposure 20 min."
    },

    {
        "facility": "Rubin",
        "instrument": "LSSTCam",
        "regime": "wide_ground",
        "observing_mode": "survey",
        "status": "operational",

        "aperture_m": 8.4,
        "fov_width_arcsec": np.nan,
        "fov_height_arcsec": np.nan,
        "fov_area_deg2": 9.6,
        "pixel_scale_arcsec": 0.20,
        "psf_fwhm_arcsec": 0.70,

        "wavelength_min_um": 0.32,
        "wavelength_max_um": 1.05,
        "gc_filter": "i",

        "survey_area_deg2": 18000.0,
        "nominal_exposure_s": 30.0,
        "survey_depth_ab": np.nan,
        "survey_depth_snr": 5.0,

        "reference": "Ivezic_2019",
        "notes": "Reference-design values; distinguish single-visit and 10-year coadded depth."
    },

    {
        "facility": "Euclid",
        "instrument": "VIS",
        "regime": "wide_space",
        "observing_mode": "survey",
        "status": "operational",

        "aperture_m": 1.2,
        "fov_width_arcsec": np.nan,
        "fov_height_arcsec": np.nan,
        "fov_area_deg2": 0.54,
        "pixel_scale_arcsec": 0.10,
        "psf_fwhm_arcsec": 0.18,

        "wavelength_min_um": 0.55,
        "wavelength_max_um": 0.90,
        "gc_filter": "VIS",

        "survey_area_deg2": 14000.0,
        "nominal_exposure_s": 4 * 565.0,
        "survey_depth_ab": 24.5,
        "survey_depth_snr": 10.0,

        "reference": "EuclidCollaboration_2025",
        "notes": "Wide-survey requirement; four nominal VIS exposures per field."
    },

    {
        "facility": "Roman",
        "instrument": "WFI",
        "regime": "wide_space",
        "observing_mode": "survey+directed",
        "status": "forthcoming",

        "aperture_m": 2.4,
        "fov_width_arcsec": np.nan,
        "fov_height_arcsec": np.nan,
        "fov_area_deg2": 0.281,
        "pixel_scale_arcsec": 0.11,
        "psf_fwhm_arcsec": np.nan,

        "wavelength_min_um": 0.48,
        "wavelength_max_um": 2.30,
        "gc_filter": "F087",  # provisional

        "survey_area_deg2": np.nan,
        "nominal_exposure_s": np.nan,
        "survey_depth_ab": np.nan,
        "survey_depth_snr": np.nan,

        "reference": "Schlieder_2024; Cromey_2025",
        "notes": "Survey depth/exposure depends on adopted Roman survey. GC filter still provisional."
    },

    {
        "facility": "ELT",
        "instrument": "MICADO + MORFEO",
        "regime": "elt_ao",
        "observing_mode": "directed",
        "status": "planned",

        "aperture_m": 39.0,
        "fov_width_arcsec": 50.5,
        "fov_height_arcsec": 50.5,
        "fov_area_deg2": (50.5 * 50.5) / 3600**2,
        "pixel_scale_arcsec": 0.004,
        "psf_fwhm_arcsec": np.nan,

        "wavelength_min_um": 0.80,
        "wavelength_max_um": 2.40,
        "gc_filter": "J",  # provisional

        "survey_area_deg2": np.nan,
        "nominal_exposure_s": np.nan,
        "survey_depth_ab": np.nan,
        "survey_depth_snr": np.nan,

        "reference": "Sturm_2024; Magrin_2024",
        "notes": "4 mas large-field imaging mode; predicted AO performance."
    },

    {
        "facility": "TMT",
        "instrument": "IRIS + NFIRAOS",
        "regime": "elt_ao",
        "observing_mode": "directed",
        "status": "planned",

        "aperture_m": 30.0,
        "fov_width_arcsec": 34.0,
        "fov_height_arcsec": 34.0,
        "fov_area_deg2": (34.0 * 34.0) / 3600**2,
        "pixel_scale_arcsec": 0.004,
        "psf_fwhm_arcsec": np.nan,

        "wavelength_min_um": 0.84,
        "wavelength_max_um": 2.40,
        "gc_filter": "J",  # provisional

        "survey_area_deg2": np.nan,
        "nominal_exposure_s": np.nan,
        "survey_depth_ab": np.nan,
        "survey_depth_snr": np.nan,

        "reference": "Larkin_2016",
        "notes": "4 mas imaging mode; predicted diffraction-limited AO performance."
    },
]

inst = pd.DataFrame(instrument_data)

inst

## Add derived geometry columns

These involve no astrophysical assumptions yet, so they’re safe to calculate now.

In [ ]:
# Useful geometric conversions
inst["fov_area_arcmin2"] = inst["fov_area_deg2"] * 3600.0

# Physical scale at a fiducial distance
D_fid_mpc = 100.0

# 1 arcsec = 4.848 * D_Mpc pc
pc_per_arcsec = 4.848 * D_fid_mpc

inst["pixel_scale_pc_100mpc"] = (
    inst["pixel_scale_arcsec"] * pc_per_arcsec
)

inst["psf_fwhm_pc_100mpc"] = (
    inst["psf_fwhm_arcsec"] * pc_per_arcsec
)

inst["fov_width_kpc_100mpc"] = (
    inst["fov_width_arcsec"] * pc_per_arcsec / 1000.0
)

inst["fov_height_kpc_100mpc"] = (
    inst["fov_height_arcsec"] * pc_per_arcsec / 1000.0
)


## Add an aperture-area proxy

We'll refer to it as a proxy, not true étendue, because we aren’t yet accounting for obscuration, throughput, vignetting, detector gaps, atmosphere, etc.


In [ ]:
inst["collecting_area_proxy_m2"] = np.pi * (inst["aperture_m"] / 2)**2

inst["etendue_proxy_m2_deg2"] = (
    inst["collecting_area_proxy_m2"] *
    inst["fov_area_deg2"]
)


## Split sensitivity into useful columns

This will be useful later, for example, single visit and coadd for comparison.

In [ ]:
for col in [
    "single_visit_depth_ab",
    "single_visit_snr",
    "coadd_depth_ab",
    "coadd_depth_snr",
    "depth_1hr_ab",
]:
    inst[col] = np.nan

In [ ]:
inst

# Add GC model machinery

Keep this deliberately modular so we can swap the turnover calibration or $\sigma$ without touching the instrument table.

A useful empirical anchor here is [Kundu & Whitmore’s (2001)](https://ui.adsabs.harvard.edu/abs/2001AJ....121.2950K/abstract) $M^\mathrm{TO}_I=-8.46\ \mathrm{mag}$; they also find $M^\mathrm{TO}_V=−7.41$. [Jordán et al (2007)](https://ui.adsabs.harvard.edu/abs/2007ApJS..171..101J/abstract). independently show that the turnover corresponds to a roughly constant characteristic mass of $∼2.2\times10^5\ \mathrm{M}\odot$ in luminous galaxies, while also demonstrating that GCLF width and, to a lesser extent, turnover mass vary with host luminosity.


In [ ]:
from scipy.stats import norm

gc_model = {
    "M_to_I": -8.46,       # empirical I-band GCLF turnover
    "sigma": 1.40,         # fiducial Gaussian width --- to be varied later
    "rh_pc": 3.0,          # representative GC half-light radius
}

M_TO = gc_model["M_to_I"]
SIGMA = gc_model["sigma"]


## Define the magnitude limits corresponding to 10%, 50%, 90%

For a Gaussian GCLF 

$$f=\Phi\left(\frac{M_\mathrm{lim}-M_\mathrm{TO}}{\sigma}\right)$$


In [ ]:
fractions = [0.10, 0.50, 0.90]

gc_limits = pd.DataFrame({
    "fraction": fractions,
})

gc_limits["z"] = norm.ppf(gc_limits["fraction"])

gc_limits["M_lim"] = (
    M_TO + SIGMA * gc_limits["z"]
)

gc_limits


The physical interpretation here is 

- to recover 10%, you only need the bright tail down to $M_I\approx−10.25$;
- 50% means reaching the turnover itself;
- 90% requires reaching almost 1.8 mag below the turnover.

Also note the slightly counterintuitive consequence:

$$D_{10}>D_{50}>D_{90}.$$


## Add the machinery the conversion of depth

For converting a depth into $D_{10/50/90}$, then for distance in Mpc

$$\mu=5\log_{10}\!D_\mathrm{Mpc} + 25,$$

and so

$$D_\mathrm{Mpc}=10^{(m_\mathrm{lim}-M_\mathrm{lim} - 25)} $$
	

In [ ]:
def distance_reach_mpc(m_lim, M_to=-8.46, sigma=1.40, fraction=0.50):
    """
    Maximum distance (Mpc) at which a limiting apparent magnitude
    recovers the specified fraction of a Gaussian GCLF.
    """
    if pd.isna(m_lim):
        return np.nan

    M_lim = M_to + sigma * norm.ppf(fraction)

    mu = m_lim - M_lim

    return 10**((mu - 25.0) / 5.0)



Quick check...

The same nominal limiting magnitude $m_\mathrm{lim}=28$mag dataset can be described as a GC survey reaching nearly 450 Mpc or only ~90 Mpc depending on what “reach” means.

In [ ]:
for f in [0.10, 0.50, 0.90]:
    print(
        f,
        distance_reach_mpc(
            m_lim=28.0,
            M_to=-8.46,
            sigma=1.40,
            fraction=f
        )
    )


## Add columns to `inst`

Prep as not populated yet

In [ ]:
inst["M_gclf_to"] = np.nan
inst["sigma_gclf"] = SIGMA

inst["D10_mpc"] = np.nan
inst["D50_mpc"] = np.nan
inst["D90_mpc"] = np.nan


# Add the GC angular-size calculation
This is independent of photometric depth, so we can calculate it immediately.

A cluster with physical half-light radius $r_h$ at $D$ Mpc subtends

$$\theta_h(^{′′})=\frac{r_h}{4.848D_\mathrm{Mpc}}.$$


In [ ]:
def gc_theta_arcsec(distance_mpc, rh_pc=3.0):
    return rh_pc / (4.848 * distance_mpc)


At Coma...


In [ ]:
theta_gc_100 = gc_theta_arcsec(100.0)

print(f"{theta_gc_100:.5f} arcsec (or ~{theta_gc_100*1e3:.1f} mas)")

So now put these against each PSF

In [ ]:
inst["gc_rh_arcsec_100mpc"] = theta_gc_100

inst["gc_resolution_ratio_100mpc"] = (
    inst["gc_rh_arcsec_100mpc"] /
    inst["psf_fwhm_arcsec"]
)


Define a ratio of angular half-light radius to instrumental PSF

$$\mathcal{R}_\mathrm{GC}=\frac{\theta_h}{\mathrm{FWHM_PSF}} $$

where small values mean strongly unresolved; approaching unity means GC structure is potentially accessible.

Additionally, we can calculate the distance at which a nominal 3-pc GC has an angular radius equal to the PSF FWHM.

In [ ]:
def gc_resolution_distance_mpc(psf_fwhm_arcsec, rh_pc=3.0):
    if pd.isna(psf_fwhm_arcsec):
        return np.nan

    return rh_pc / (4.848 * psf_fwhm_arcsec)


inst["D_gc_rh_equals_psf_mpc"] = (
    inst["psf_fwhm_arcsec"]
    .apply(gc_resolution_distance_mpc)
)


# Band-pass normalization

Before proceeding, we need to make sure `M_gclf_to` band-specific

So, for an immediate proof-of-concept plot, use the same empirical

$$M_I^\mathrm{TO}=−8.46$$

for the approximately $I$-like instruments only:

In [ ]:
optical_first_pass = {
    "HST":    -8.46,   # F814W approximation
    "Subaru": -8.46,   # i approximation
    "Rubin":  -8.46,   # i approximation
}


In [ ]:
gclf_filters = pd.DataFrame({
    "facility": [
        "HST", "JWST", "Subaru", "Rubin",
        "Euclid", "Roman", "ELT", "TMT"
    ],
    "filter": [
        "F814W", "F150W", "i", "i",
        "VIS", "F087", "J", "J"
    ],
    "lambda_eff_um": [
        np.nan, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan, np.nan
    ],
    "M_gclf_to": [
        np.nan, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan, np.nan
    ],
    "M_to_source": [
        "", "", "", "", "", "", "", ""
    ]
})


# Add separate set of filter-specific columns
We’ll need these for the GCLF modelling:

In [ ]:
inst["gc_filter_lambda_eff_um"] = np.nan
inst["gc_filter_lambda_min_um"] = np.nan
inst["gc_filter_lambda_max_um"] = np.nan


In [ ]:
inst.loc[inst["facility"] == "Roman", "gc_filter"] # = "F087"

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        inst
    )

## Populate filter specific columns

Only for a feasibility plot, these are the initial 'first-look' values, from published/official pivot or central wavelengths where available, and quoted blue/red edges or a simple FWHM-style approximation otherwise.

[JWST](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-filters#gsc.tab=0) and [Roman](https://roman-docs.stsci.edu/roman-instruments/the-wide-field-instrument/observing-with-the-wfi/wfi-optical-elements) literature give exactly the values we need, but the others are estimates.

In [ ]:
filter_firstpass = {
    "HST": {
        "filter": "F814W",
        "lambda_eff_um": 0.806,
        "lambda_min_um": 0.73,
        "lambda_max_um": 0.88,
        "basis": "published pivot + approximate broadband limits",
    },

    "JWST": {
        "filter": "F150W",
        "lambda_eff_um": 1.501,
        "lambda_min_um": 1.331,
        "lambda_max_um": 1.668,
        "basis": "STScI commissioning total-system throughput",
    },

    "Subaru": {
        "filter": "i2",
        "lambda_eff_um": 0.77,
        "lambda_min_um": 0.70,
        "lambda_max_um": 0.85,
        "basis": "HSC published filter curve; first-pass limits",
    },

    "Rubin": {
        "filter": "i",
        "lambda_eff_um": 0.75,
        "lambda_min_um": 0.69,
        "lambda_max_um": 0.82,
        "basis": "LSST total-system band approximation",
    },

    "Euclid": {
        "filter": "VIS",
        "lambda_eff_um": 0.72,
        "lambda_min_um": 0.55,
        "lambda_max_um": 0.90,
        "basis": "Euclid VIS nominal broad passband",
    },

    "Roman": {
        "filter": "F087",
        "lambda_eff_um": 0.8696,
        "lambda_min_um": 0.795,
        "lambda_max_um": 0.944,
        "basis": "Roman WFI pivot and FWHM",
    },

    "ELT": {
        "filter": "J",
        "lambda_eff_um": 1.25,
        "lambda_min_um": 1.15,
        "lambda_max_um": 1.35,
        "basis": "standard J-band approximation",
    },

    "TMT": {
        "filter": "J",
        "lambda_eff_um": 1.25,
        "lambda_min_um": 1.15,
        "lambda_max_um": 1.35,
        "basis": "standard J-band approximation",
    },
}

In [ ]:
# %conda install stsynphot -c conda-forge

# persue this later if the initial plots look promising.

In [ ]:
# import stsynphot as STS # a new and improved replace for pysynphot
# import matplotlib.pyplot as plt

# obsmode = 'acs,wfc1,f814w' # desired instrument, chip1, filter, and date

# # Instantiate an ObservationSpectralElement object which contains the desired
# # information (throughput, wavelength converage, etc..) about the specified obsmode
# bp_acs = STS.band(obsmode) 

# # Now we get the wavelengths associated with the throughput curve.
# wavelengths = bp_acs.binset 

# # One of the main features of stsynphot is that it is built using astropy.units
# # This means that all values are returned with their correct units, so no more
# # wondering what the units are of the values you have!
# print(wavelengths.unit)

# # Finally, compute the throughput values for the filter over the provided wavelength range
# throughput = bp_acs(wavelengths) 

# # Check to make sure everything looks good
# plt.plot(wavelengths.value, throughput.value, label='F555W')

In [ ]:
# import os
# import stsynphot as STS

# print("PYSYN_CDBS =", os.environ.get("PYSYN_CDBS"))
# STS.showref()